# Week 1: How Neural Networks Transform Space

A neural network doesn't just "learn a function" — it **warps and reshapes the input space** until data points become linearly separable.

In this notebook, we'll visualize exactly what happens inside a neural network, layer by layer.

## 0. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.datasets import make_circles

torch.manual_seed(42)
np.random.seed(42)

# Nice colors
COLORS = ['#FF6B6B', '#4ECDC4']
CMAP = ListedColormap(COLORS)

---
## 1. The Problem: Non-Linearly Separable Data

Some datasets simply **cannot** be separated by a straight line. Let's see two examples.

In [ ]:
# Generate datasets
X_circles, y_circles = make_circles(n_samples=500, noise=0.05, factor=0.5)

# Spiral dataset
def make_spirals(n_samples=500, noise=0.5):
    n = n_samples // 2
    theta = np.linspace(0, 3 * np.pi, n)
    r = np.linspace(0.5, 3, n)
    x1 = np.column_stack([r * np.cos(theta) + np.random.randn(n) * noise * 0.1,
                          r * np.sin(theta) + np.random.randn(n) * noise * 0.1])
    x2 = np.column_stack([-r * np.cos(theta) + np.random.randn(n) * noise * 0.1,
                          -r * np.sin(theta) + np.random.randn(n) * noise * 0.1])
    X = np.vstack([x1, x2])
    y = np.hstack([np.zeros(n), np.ones(n)])
    return X, y.astype(int)

X_spirals, y_spirals = make_spirals(500, noise=0.5)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, X, y, title in zip(axes,
                            [X_circles, X_spirals],
                            [y_circles, y_spirals],
                            ['Concentric Circles', 'Spirals']):
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap=CMAP, s=15, alpha=0.8)
    ax.set_title(title, fontsize=14)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
plt.suptitle('Can you draw a single straight line to separate the red and blue points?', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## 2. A Linear Classifier Fails

A single linear layer (no activation) can only draw a straight line. Let's see it try and fail on the circles dataset.

In [ ]:
def plot_decision_boundary(model, X, y, ax=None, title=''):
    """Plot the decision boundary of a model on 2D data."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 5))
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))
    grid = torch.FloatTensor(np.c_[xx.ravel(), yy.ravel()])
    with torch.no_grad():
        preds = model(grid).squeeze()
        if preds.dim() > 1:
            preds = preds[:, 1] - preds[:, 0]
        preds = torch.sigmoid(preds).numpy().reshape(xx.shape)
    ax.contourf(xx, yy, preds, levels=50, cmap='RdYlGn', alpha=0.4)
    ax.contour(xx, yy, preds, levels=[0.5], colors='black', linewidths=2)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap=CMAP, s=15, edgecolors='k', linewidths=0.3)
    ax.set_title(title, fontsize=13)
    ax.set_aspect('equal')
    return ax


def train_model(model, X, y, epochs=1000, lr=0.1):
    """Train a binary classifier."""
    X_t = torch.FloatTensor(X)
    y_t = torch.FloatTensor(y)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()
    for epoch in range(epochs):
        out = model(X_t).squeeze()
        loss = criterion(out, y_t)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    acc = ((torch.sigmoid(out) > 0.5).float() == y_t).float().mean()
    return loss.item(), acc.item()


# Linear model: just one layer, no activation
linear_model = nn.Linear(2, 1)
loss, acc = train_model(linear_model, X_circles, y_circles)

fig, ax = plt.subplots(figsize=(6, 5))
plot_decision_boundary(linear_model, X_circles, y_circles, ax,
                       f'Linear Classifier (Acc: {acc:.1%})')
plt.tight_layout()
plt.show()
print('A straight line simply cannot separate concentric circles!')

---
## 3. The Key Idea: Neural Networks Warp Space

A neural network with hidden layers doesn't just draw a complex boundary — it **transforms the coordinate system** so that a simple line *can* separate the data.

Let's build a 2D → 2D → 1 network so we can **visualize the hidden space**.

In [ ]:
class WarpNet(nn.Module):
    """2D -> 2D -> 1  (we can visualize the 2D hidden space!)"""
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(2, 2)    # 2D input -> 2D hidden
        self.act1 = nn.ReLU()
        self.layer2 = nn.Linear(2, 1)    # 2D hidden -> 1D output

    def forward(self, x):
        x = self.layer1(x)
        x = self.act1(x)
        x = self.layer2(x)
        return x

    def get_hidden(self, x):
        """Return the 2D hidden representation."""
        with torch.no_grad():
            x = self.layer1(x)
            x = self.act1(x)
        return x.numpy()


warp_net = WarpNet()
loss, acc = train_model(warp_net, X_circles, y_circles, epochs=2000, lr=0.01)

X_t = torch.FloatTensor(X_circles)
H = warp_net.get_hidden(X_t)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Original space
axes[0].scatter(X_circles[:, 0], X_circles[:, 1], c=y_circles, cmap=CMAP, s=20, edgecolors='k', linewidths=0.3)
axes[0].set_title('Input Space (Original)', fontsize=13)
axes[0].set_xlabel('$x_1$'); axes[0].set_ylabel('$x_2$')
axes[0].set_aspect('equal'); axes[0].grid(True, alpha=0.3)

# Hidden space (after layer1 + ReLU)
axes[1].scatter(H[:, 0], H[:, 1], c=y_circles, cmap=CMAP, s=20, edgecolors='k', linewidths=0.3)
axes[1].set_title('Hidden Space (After Layer 1 + ReLU)', fontsize=13)
axes[1].set_xlabel('$h_1$'); axes[1].set_ylabel('$h_2$')
axes[1].set_aspect('equal'); axes[1].grid(True, alpha=0.3)

# Decision boundary in original space
plot_decision_boundary(warp_net, X_circles, y_circles, axes[2],
                       f'Decision Boundary (Acc: {acc:.1%})')

plt.suptitle('The hidden layer WARPS space so a line can separate the classes!', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 4. Grid Deformation: Watching Space Bend

To really *see* the warping, let's pass a **uniform grid** through the network and watch how it gets deformed.

In [ ]:
def plot_grid_transformation(model, X, y, n_grid=20):
    """Show how a uniform grid is warped by the hidden layer."""
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5

    # Create grid
    xs = np.linspace(x_min, x_max, n_grid)
    ys = np.linspace(y_min, y_max, n_grid)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # -- Original grid --
    ax = axes[0]
    for x_val in xs:
        ax.plot([x_val]*n_grid, ys, 'b-', alpha=0.2, linewidth=0.8)
    for y_val in ys:
        ax.plot(xs, [y_val]*n_grid, 'b-', alpha=0.2, linewidth=0.8)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap=CMAP, s=20, edgecolors='k', linewidths=0.3, zorder=5)
    ax.set_title('Original Space (Uniform Grid)', fontsize=13)
    ax.set_aspect('equal'); ax.grid(False)

    # -- Warped grid --
    ax = axes[1]
    # Transform horizontal lines
    for y_val in ys:
        line_pts = np.column_stack([xs, np.full(n_grid, y_val)])
        line_t = torch.FloatTensor(line_pts)
        warped = model.get_hidden(line_t)
        ax.plot(warped[:, 0], warped[:, 1], 'b-', alpha=0.2, linewidth=0.8)
    # Transform vertical lines
    for x_val in xs:
        line_pts = np.column_stack([np.full(n_grid, x_val), ys])
        line_t = torch.FloatTensor(line_pts)
        warped = model.get_hidden(line_t)
        ax.plot(warped[:, 0], warped[:, 1], 'b-', alpha=0.2, linewidth=0.8)
    # Transform data points
    H = model.get_hidden(torch.FloatTensor(X))
    ax.scatter(H[:, 0], H[:, 1], c=y, cmap=CMAP, s=20, edgecolors='k', linewidths=0.3, zorder=5)
    ax.set_title('Hidden Space (Warped Grid)', fontsize=13)
    ax.set_aspect('equal'); ax.grid(False)

    plt.suptitle('The network bends, folds, and stretches space!', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()


plot_grid_transformation(warp_net, X_circles, y_circles)

---
## 5. With vs. Without Activation Function

What happens if we remove the activation (ReLU)? Without it, stacking layers is just matrix multiplication — the result is still a **linear** transformation. No warping, no bending.

In [ ]:
class LinearOnlyNet(nn.Module):
    """Same architecture but NO activation function."""
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(2, 2)
        # No activation!
        self.layer2 = nn.Linear(2, 1)

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        return x

    def get_hidden(self, x):
        with torch.no_grad():
            x = self.layer1(x)
        return x.numpy()


class NonLinearNet(nn.Module):
    """Same architecture WITH activation."""
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(2, 2)
        self.act1 = nn.ReLU()
        self.layer2 = nn.Linear(2, 1)

    def forward(self, x):
        x = self.layer1(x)
        x = self.act1(x)
        x = self.layer2(x)
        return x

    def get_hidden(self, x):
        with torch.no_grad():
            x = self.layer1(x)
            x = self.act1(x)
        return x.numpy()


lin_net = LinearOnlyNet()
nonlin_net = NonLinearNet()

loss_l, acc_l = train_model(lin_net, X_circles, y_circles, epochs=2000, lr=0.01)
loss_n, acc_n = train_model(nonlin_net, X_circles, y_circles, epochs=2000, lr=0.01)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Top row: without activation
H_lin = lin_net.get_hidden(torch.FloatTensor(X_circles))
axes[0, 0].scatter(H_lin[:, 0], H_lin[:, 1], c=y_circles, cmap=CMAP, s=15, edgecolors='k', linewidths=0.3)
axes[0, 0].set_title('Hidden Space — NO Activation', fontsize=13)
axes[0, 0].set_aspect('equal'); axes[0, 0].grid(True, alpha=0.3)
plot_decision_boundary(lin_net, X_circles, y_circles, axes[0, 1],
                       f'Without Activation (Acc: {acc_l:.1%})')

# Bottom row: with activation
H_nonlin = nonlin_net.get_hidden(torch.FloatTensor(X_circles))
axes[1, 0].scatter(H_nonlin[:, 0], H_nonlin[:, 1], c=y_circles, cmap=CMAP, s=15, edgecolors='k', linewidths=0.3)
axes[1, 0].set_title('Hidden Space — WITH ReLU', fontsize=13)
axes[1, 0].set_aspect('equal'); axes[1, 0].grid(True, alpha=0.3)
plot_decision_boundary(nonlin_net, X_circles, y_circles, axes[1, 1],
                       f'With ReLU (Acc: {acc_n:.1%})')

plt.suptitle('Activation functions enable non-linear space warping', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

---
## 6. Going Deeper: Layer-by-Layer Transformation

Now let's build a deeper network (4 hidden layers, all 2D) and watch data points travel through **each layer**.

In [ ]:
class DeepWarpNet(nn.Module):
    """2D -> 2D -> 2D -> 2D -> 2D -> 1"""
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(2, 2),
            nn.Linear(2, 2),
            nn.Linear(2, 2),
            nn.Linear(2, 2),
        ])
        self.output = nn.Linear(2, 1)
        self.act = nn.ReLU()

    def forward(self, x):
        for layer in self.layers:
            x = self.act(layer(x))
        return self.output(x)

    def get_all_hidden(self, x):
        """Return representations after each layer."""
        hiddens = [x.numpy().copy()]
        with torch.no_grad():
            for layer in self.layers:
                x = self.act(layer(x))
                hiddens.append(x.numpy().copy())
        return hiddens


deep_net = DeepWarpNet()
loss, acc = train_model(deep_net, X_circles, y_circles, epochs=3000, lr=0.01)

X_t = torch.FloatTensor(X_circles)
hiddens = deep_net.get_all_hidden(X_t)

fig, axes = plt.subplots(1, 6, figsize=(24, 4))
titles = ['Input', 'After Layer 1', 'After Layer 2', 'After Layer 3', 'After Layer 4', 'Decision Boundary']

for i, (ax, H, title) in enumerate(zip(axes[:-1], hiddens, titles[:-1])):
    ax.scatter(H[:, 0], H[:, 1], c=y_circles, cmap=CMAP, s=10, edgecolors='k', linewidths=0.2)
    ax.set_title(title, fontsize=12)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

plot_decision_boundary(deep_net, X_circles, y_circles, axes[-1],
                       f'Decision Boundary\n(Acc: {acc:.1%})')

plt.suptitle('Layer-by-layer: watch the data points reorganize until they are linearly separable',
             fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

---
## 7. Width Matters: More Neurons = More Expressive

A wider hidden layer gives the network more "room" to rearrange points. Let's compare 2D, 8D, and 32D hidden layers on the harder **circles** dataset.

(For >2D hidden layers we can't directly visualize, so we show the decision boundary instead.)

In [ ]:
class VariableWidthNet(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.net(x)


fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, width in zip(axes, [2, 8, 32]):
    model = VariableWidthNet(width)
    loss, acc = train_model(model, X_circles, y_circles, epochs=3000, lr=0.01)
    plot_decision_boundary(model, X_circles, y_circles, ax,
                           f'Hidden dim = {width} (Acc: {acc:.1%})')

plt.suptitle('Wider networks can learn more complex boundaries', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 8. Depth Matters: More Layers = More Folds

Each layer can introduce one more "fold" in the space. Let's compare 1, 2, and 4 hidden layers on the **spiral** dataset.

In [ ]:
class VariableDepthNet(nn.Module):
    def __init__(self, n_layers, hidden_dim=32):
        super().__init__()
        layers = [nn.Linear(2, hidden_dim), nn.ReLU()]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.ReLU()]
        layers.append(nn.Linear(hidden_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, depth in zip(axes, [1, 2, 4]):
    model = VariableDepthNet(n_layers=depth, hidden_dim=32)
    loss, acc = train_model(model, X_spirals, y_spirals, epochs=5000, lr=0.01)
    plot_decision_boundary(model, X_spirals, y_spirals, ax,
                           f'{depth} hidden layer{"s" if depth > 1 else ""} (Acc: {acc:.1%})')

plt.suptitle('Deeper networks can separate more complex patterns', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 9. Activation Function Comparison

Different activations warp space differently. Let's compare **ReLU**, **Tanh**, and **Sigmoid**.

In [ ]:
class ActivationCompareNet(nn.Module):
    def __init__(self, activation):
        super().__init__()
        self.layer1 = nn.Linear(2, 2)
        self.act = activation
        self.layer2 = nn.Linear(2, 1)

    def forward(self, x):
        x = self.act(self.layer1(x))
        return self.layer2(x)

    def get_hidden(self, x):
        with torch.no_grad():
            x = self.act(self.layer1(x))
        return x.numpy()


activations = [
    ('ReLU', nn.ReLU()),
    ('Tanh', nn.Tanh()),
    ('Sigmoid', nn.Sigmoid()),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for col, (name, act) in enumerate(activations):
    model = ActivationCompareNet(act)
    loss, acc = train_model(model, X_circles, y_circles, epochs=2000, lr=0.01)
    H = model.get_hidden(torch.FloatTensor(X_circles))

    # Hidden space
    axes[0, col].scatter(H[:, 0], H[:, 1], c=y_circles, cmap=CMAP, s=15, edgecolors='k', linewidths=0.2)
    axes[0, col].set_title(f'{name} — Hidden Space', fontsize=13)
    axes[0, col].set_aspect('equal'); axes[0, col].grid(True, alpha=0.3)

    # Decision boundary
    plot_decision_boundary(model, X_circles, y_circles, axes[1, col],
                           f'{name} — Boundary (Acc: {acc:.1%})')

plt.suptitle('Different activations warp space in different ways', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

---
## 10. Training Animation: Watching the Network Learn

Let's watch how the hidden space and decision boundary **evolve during training**.

In [ ]:
class AnimNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(2, 2)
        self.act = nn.ReLU()
        self.layer2 = nn.Linear(2, 1)

    def forward(self, x):
        return self.layer2(self.act(self.layer1(x)))

    def get_hidden(self, x):
        with torch.no_grad():
            return self.act(self.layer1(x)).numpy()


model = AnimNet()
X_t = torch.FloatTensor(X_circles)
y_t = torch.FloatTensor(y_circles)
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.BCEWithLogitsLoss()

snapshots = [0, 10, 50, 200, 500, 2000]
fig, axes = plt.subplots(2, len(snapshots), figsize=(4 * len(snapshots), 8))

snap_idx = 0
for epoch in range(2001):
    out = model(X_t).squeeze()
    loss = criterion(out, y_t)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if snap_idx < len(snapshots) and epoch == snapshots[snap_idx]:
        acc = ((torch.sigmoid(out) > 0.5).float() == y_t).float().mean().item()

        # Hidden space
        H = model.get_hidden(X_t)
        axes[0, snap_idx].scatter(H[:, 0], H[:, 1], c=y_circles, cmap=CMAP, s=10,
                                  edgecolors='k', linewidths=0.2)
        axes[0, snap_idx].set_title(f'Epoch {epoch}', fontsize=11)
        axes[0, snap_idx].set_aspect('equal'); axes[0, snap_idx].grid(True, alpha=0.3)

        # Decision boundary
        plot_decision_boundary(model, X_circles, y_circles, axes[1, snap_idx],
                               f'Acc: {acc:.1%}')
        snap_idx += 1

axes[0, 0].set_ylabel('Hidden Space', fontsize=12)
axes[1, 0].set_ylabel('Decision Boundary', fontsize=12)
plt.suptitle('Training progression: the network gradually learns to warp space', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## Key Takeaways

1. **Without activation functions**, stacking layers is equivalent to a single linear transformation — you can only draw straight lines.

2. **Activation functions** (ReLU, Tanh, etc.) allow the network to **fold, stretch, and warp** the input space.

3. Each hidden layer adds another **non-linear transformation** — the data gets progressively reshaped until classes are **linearly separable**.

4. **Wider** layers give more dimensions to spread data apart. **Deeper** networks can fold space more times, handling more complex patterns.

5. Training is the process of **learning the right warping** — adjusting weights so the transformation makes the final linear classifier's job easy.